In [9]:
import os
import pickle
import pandas as pd
import numpy as np
import scipy.signal as signal
from scipy import stats
from sklearn.preprocessing import MinMaxScaler

In [10]:
# --- HELPER: Calculate Rolling HRV ---
def calculate_rolling_hrv(bvp_signal, fs=64, window_sec=60):
    """Computes rolling RMSSD (Heart Rate Variability) from BVP."""
    peaks, _ = signal.find_peaks(bvp_signal, distance=fs/2.5, height=np.mean(bvp_signal))
    peak_times = peaks / fs  
    ibi = np.diff(peak_times) * 1000 
    ibi_series = pd.Series(ibi, index=pd.to_timedelta(peak_times[1:], unit='s'))
    diff_ibi_sq = ibi_series.diff() ** 2
    rmssd_series = diff_ibi_sq.rolling(window=f'{window_sec}s', min_periods=10).mean().pow(0.5)
    return rmssd_series

# --- UPDATED: Process Single Subject ---
def align_subject_data(wrist_data, labels_data, target_freq=64):
    """Aligns sensors and calculates HRV for a SINGLE subject."""
    
    bvp = wrist_data['BVP'].flatten()  # 64 Hz
    acc = wrist_data['ACC']            # 32 Hz
    temp = wrist_data['TEMP'].flatten() # 4 Hz
    labels = labels_data               # 700 Hz

    # 1. BVP (The Anchor - 64Hz)
    idx_bvp = pd.to_timedelta(np.arange(len(bvp)) / target_freq, unit='s')
    df_bvp = pd.DataFrame(bvp, index=idx_bvp, columns=['BVP'])

    # 2. ACC (32Hz -> 64Hz)
    idx_acc = pd.to_timedelta(np.arange(len(acc)) / 32, unit='s')
    df_acc = pd.DataFrame(acc, index=idx_acc, columns=['ACC_x', 'ACC_y', 'ACC_z'])
    df_acc = df_acc.resample(f'{1/target_freq}s').interpolate(method='linear')

    # 3. TEMP (4Hz -> 64Hz)
    idx_temp = pd.to_timedelta(np.arange(len(temp)) / 4, unit='s')
    df_temp = pd.DataFrame(temp, index=idx_temp, columns=['TEMP'])
    df_temp = df_temp.resample(f'{1/target_freq}s').interpolate(method='linear')

    # 4. Labels (700Hz -> 64Hz)
    idx_label = pd.to_timedelta(np.arange(len(labels)) / 700, unit='s')
    df_label = pd.DataFrame(labels, index=idx_label, columns=['label'])
    df_label = df_label.resample(f'{1/target_freq}s').nearest()

    # 5. HRV Extraction
    hrv_sparse = calculate_rolling_hrv(df_bvp['BVP'].values, fs=target_freq, window_sec=60)
    df_hrv = pd.DataFrame(hrv_sparse, columns=['HRV_RMSSD'])
    df_hrv = df_hrv.reindex(df_bvp.index).ffill().bfill()

    # Merge and drop NaNs (the first 60 seconds of this specific subject)
    df_main = pd.concat([df_acc, df_bvp, df_temp, df_hrv, df_label], axis=1)
    df_main.dropna(inplace=True) 
    
    return df_main

In [11]:
# --- UNCHANGED: Create Windows ---
def create_windows(df, window_size=256, step=128):
    """Slices continuous data into overlapping windows."""
    features = df.iloc[:, :-1].values
    labels = df.iloc[:, -1].values
    
    X, y = [], []
    for i in range(0, len(df) - window_size, step):
        window = features[i : i + window_size]
        label_window = labels[i : i + window_size]
        label_mode = stats.mode(label_window, keepdims=True)[0][0]
        X.append(window)
        y.append(label_mode)
        
    return np.array(X), np.array(y)

In [12]:
# --- NEW: Master Processing Loop ---
if __name__ == "__main__":
    base_path = 'data/WESAD'
    
    if not os.path.exists(base_path):
        raise FileNotFoundError(f"Directory not found: {base_path}")

    # Find all subject folders (S2, S3, S4, etc.)
    subject_folders = [f for f in os.listdir(base_path) if f.startswith('S') and os.path.isdir(os.path.join(base_path, f))]
    subject_folders.sort(key=lambda x: int(x[1:])) 

    print(f"Found {len(subject_folders)} subjects to process: {subject_folders}\n")

    master_X = []
    master_y = []

    for subj in subject_folders:
        print(f"--- Processing Subject {subj} ---")
        # 1. Load Data
        # ... (Loading logic stays the same) ...

        # 2. Align & Extract Features
        df_subj = align_subject_data(wrist_data, labels_data)

        # 3. Filter Labels
        df_subj = df_subj[df_subj['label'].isin([1, 2, 3, 4])]

        # ==========================================================
        # --- NEW: PER-SUBJECT NORMALIZATION ---
        # ==========================================================
        scaler = MinMaxScaler()
        
        # Find ONLY this patient's baseline data to learn what is "Normal" for them
        baseline_mask = df_subj['label'].isin([1, 4])
        baseline_data = df_subj[baseline_mask].iloc[:, :-1] # All feature columns
        
        if not baseline_data.empty:
            scaler.fit(baseline_data)
            
            # Now apply that personal baseline scale to ALL their data (including stress)
            # This makes 1.0 = "Their personal maximum normal level"
            df_subj.iloc[:, :-1] = scaler.transform(df_subj.iloc[:, :-1])
        # ==========================================================

        # 4. Windowing (Data is now scaled!)
        X_subj, y_subj = create_windows(df_subj, window_size=256, step=128)
        
        print(f"  Extracted {len(X_subj)} windows.")
        
        # 5. Append to Master List
        master_X.append(X_subj)
        master_y.append(y_subj)

    print("\n--- Finalizing Dataset ---")
    # Stack all subject windows together into one giant array
    X_final = np.vstack(master_X)
    y_final = np.concatenate(master_y)

    print(f"Total X Shape: {X_final.shape} (Windows, Time Steps, Features)")
    print(f"Total y Shape: {y_final.shape} (Labels)")
    
    print(f"Class Distribution: {np.unique(y_final, return_counts=True)}")

    np.save('X_data_ALL.npy', X_final)
    np.save('y_data_ALL.npy', y_final)
    print("All patient data successfully saved to 'X_data_ALL.npy' and 'y_data_ALL.npy'")

Found 15 subjects to process: ['S2', 'S3', 'S4', 'S5', 'S6', 'S7', 'S8', 'S9', 'S10', 'S11', 'S13', 'S14', 'S15', 'S16', 'S17']

--- Processing Subject S2 ---
  Extracted 1502 windows.
--- Processing Subject S3 ---
  Extracted 1502 windows.
--- Processing Subject S4 ---
  Extracted 1502 windows.
--- Processing Subject S5 ---
  Extracted 1502 windows.
--- Processing Subject S6 ---
  Extracted 1502 windows.
--- Processing Subject S7 ---
  Extracted 1502 windows.
--- Processing Subject S8 ---
  Extracted 1502 windows.
--- Processing Subject S9 ---
  Extracted 1502 windows.
--- Processing Subject S10 ---
  Extracted 1502 windows.
--- Processing Subject S11 ---
  Extracted 1502 windows.
--- Processing Subject S13 ---
  Extracted 1502 windows.
--- Processing Subject S14 ---
  Extracted 1502 windows.
--- Processing Subject S15 ---
  Extracted 1502 windows.
--- Processing Subject S16 ---
  Extracted 1502 windows.
--- Processing Subject S17 ---
  Extracted 1502 windows.

--- Finalizing Dataset 